In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.3_signlessCorr_comparison_onlyMat.py
------------------------------------------------
4.2.4.3 符号無視相関（signless）との比較：new（Main）vs signless（Abs-corr）
- 既存の ClusterAssign_*（new）と、signless の cluster_labels_* を (mode,index) で突き合わせ
- サンプルID（id）に基づき、各 (mode,index) のクラスタ割当の一致度（ARI/NMI/AMI）を算出
- k（クラスタ数）と n（共通ID数）も追記
- 本文（k≤30）と補遺（all k）を自動で分割して出力
- 図：ヒートマップ（ARI）に (kA/kB) をセル内表示、背景輝度で文字色自動調整
- 出力フォルダ/ファイル名は 4.2.4.3 を含み、本文/補遺が分かる命名

入力期待：
  ROOT/
    ├─ figs_OH/ClusterAssign_(top3|cumeig)_(index)_OH_*.csv
    ├─ figs_FP/ClusterAssign_(top3|cumeig)_(index)_FP_*.csv
    ├─ STEP3.2_signlessCorr_MDS_YYYYmmdd/{OH,FP}/cluster_labels_*.csv  ← signless
    └─ （features_*onlyMat*.csv は使用しませんが同階層に存在していてもOK）

出力：
  ROOT/paper_4.2.4.3_signless/
    ├─ main_text/
    │   ├─ Table_4.2.4.3_new-vs-signless_k-le30.csv
    │   └─ Fig_4.2.4.3_new-vs-signless_ARI-heatmap.(png|pdf)
    ├─ appendix/
    │   ├─ Appendix_Table_allK_new-vs-signless.csv
    │   └─ Appendix_Fig_allK_new-vs-signless_ARI-heatmap.(png|pdf)
    └─ analysis_csv/
        └─ Table_4.2.4.3_k-by-index.csv  # 参考：new側のk一覧
"""

from pathlib import Path
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.colors import to_rgb
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score

# ========= ユーザー設定（必要に応じて変更） =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_DIR = {"OH": ROOT / "figs_OH", "FP": ROOT / "figs_FP"}

# signless の所在（STEP名は環境に合わせて）
SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"  # 例：…/STEP3.2_signlessCorr_MDS_20250904/{OH,FP}/cluster_labels_*.csv
SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

OUT_ROOT     = ROOT / "paper_4.2.4.3_signless"
OUT_MAIN     = OUT_ROOT / "main_text"
OUT_APPENDIX = OUT_ROOT / "appendix"
OUT_ANALYSIS = OUT_ROOT / "analysis_csv"
for d in [OUT_ROOT, OUT_MAIN, OUT_APPENDIX, OUT_ANALYSIS]:
    d.mkdir(parents=True, exist_ok=True)

DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
K_MAX_KEEP = 30
DPI = 300

# ========= フォント（日本語フォールバック） =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams.update({"font.size":10, "axes.titlesize":12, "axes.labelsize":10, "legend.fontsize":9})

# ========= IO =========
def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

# new (Main) 側：ClusterAssign_(top3|cumeig)_(index)_{ds}_*.csv をマージ（id列つき）
FN_MAIN = re.compile(r"^ClusterAssign_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)_(OH|FP)_.*\.csv$")
def collect_new_labels(figs_dir: Path, ds: str) -> pd.DataFrame | None:
    files = [p for p in figs_dir.glob("ClusterAssign_*.csv") if FN_MAIN.match(p.name)]
    parts = []
    for f in files:
        m = re.match(r"ClusterAssign_(top3|cumeig)_([A-Za-z]+)_(OH|FP)_", f.name)
        if not m or m.group(3) != ds: continue
        mode, index = m.group(1), m.group(2)
        df = read_csv_quiet(f)
        if df is None: continue
        if {"Variable","Cluster"}.issubset(df.columns):
            ids = df["Variable"].astype(str).values
            cl  = df["Cluster"].values
        elif df.shape[1] == 1:
            ids = df.index.astype(str).values
            cl  = df.iloc[:,0].values
        else:
            continue
        cond = f"{mode}_{index}"
        codes = pd.Categorical(cl).codes
        codes = pd.Series(codes).replace({-1: np.nan}).values
        parts.append(pd.DataFrame({"id": ids, cond: codes}))
    if not parts: return None
    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on="id", how="outer")
    return out.drop_duplicates(subset=["id"])

# signless 側：cluster_labels_*.csv の linear_* 列だけ抽出（id必須）
def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
    cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
    if not cands: return None
    df = read_csv_quiet(cands[0])
    if df is None or "id" not in df.columns: return None
    keep = [c for c in df.columns if str(c).startswith("linear_")]
    if not keep: return None
    out = df[["id"]+keep].copy()
    out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]  # → "top3_silhouette" 等
    return out

# ========= 計算 =========
def pairwise_scores(dfA: pd.DataFrame|None, dfB: pd.DataFrame|None) -> pd.DataFrame:
    cols = ["condition","n","kA","kB","ARI","NMI","AMI"]
    if dfA is None or dfB is None: return pd.DataFrame(columns=cols)
    colsA = [c for c in dfA.columns if c!="id"]
    colsB = [c for c in dfB.columns if c!="id"]
    common = sorted(set(colsA).intersection(colsB))
    if not common: return pd.DataFrame(columns=cols)
    merged = dfA.merge(dfB, on="id", suffixes=(".A",".B"))
    rows=[]
    for cn in common:
        a = merged[f"{cn}.A"].values
        b = merged[f"{cn}.B"].values
        mask = ~pd.isna(a) & ~pd.isna(b)
        if mask.sum() < 2: continue
        a = a[mask].astype(int); b = b[mask].astype(int)
        rows.append([
            cn, int(mask.sum()),
            int(len(np.unique(a))), int(len(np.unique(b))),
            float(adjusted_rand_score(a,b)),
            float(normalized_mutual_info_score(a,b)),
            float(adjusted_mutual_info_score(a,b)),
        ])
    return pd.DataFrame(rows, columns=cols)

def summarize_k_from_assign(figs_dir: Path, ds: str) -> pd.DataFrame:
    rows=[]
    for mode in DIMS:
        for ix in INDICES:
            pat = f"ClusterAssign_{mode}_{ix}_{ds}_*.csv"
            matches = sorted(figs_dir.glob(pat), key=os.path.getmtime, reverse=True)
            if not matches:
                rows.append([ds, mode, ix, np.nan]); continue
            df = read_csv_quiet(matches[0])
            if df is None:
                rows.append([ds, mode, ix, np.nan]); continue
            if {"Variable","Cluster"}.issubset(df.columns):
                k = pd.Series(df["Cluster"]).nunique(dropna=True)
            elif df.shape[1] == 1:
                k = pd.Series(df.iloc[:,0]).nunique(dropna=True)
            else:
                k = np.nan
            rows.append([ds, mode, ix, k])
    return pd.DataFrame(rows, columns=["Dataset","mode","index","k"])

# ========= 図 =========
def _light_or_dark_text(rgb_tuple):
    r,g,b = rgb_tuple
    L = 0.2126*r + 0.7152*g + 0.0722*b
    return "black" if L > 0.6 else "white"

def save_heatmap_ari(df_pivot: pd.DataFrame, meta_table: pd.DataFrame, out_path_png: Path, out_path_pdf: Path, title: str, subtitle: str):
    set_font()
    Z = df_pivot.values.astype(float)
    row_labels = list(df_pivot.index)           # ["OH new vs signless", "FP new vs signless"]
    col_labels = list(df_pivot.columns)         # e.g., "top3–silhouette"...

    # (kA/kB) 取得用メタ（condition は "top3_silhouette" 等）
    meta = meta_table.copy()
    meta["key"] = meta["condition"].str.replace("_", "–", regex=False)
    meta_idx = {(r["Dataset"], r["key"]): (r["kA"], r["kB"]) for _, r in meta.iterrows()}

    fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)
    im = ax.imshow(Z, vmin=0, vmax=1, cmap="Blues", aspect="auto")

    ax.set_xticks(np.arange(Z.shape[1])); ax.set_yticks(np.arange(Z.shape[0]))
    ax.set_xticklabels(col_labels, rotation=35, ha="right")
    ax.set_yticklabels(row_labels)

    for i, ds in enumerate(df_pivot.index):
        for j, key in enumerate(df_pivot.columns):
            val = Z[i, j]
            if np.isnan(val): continue
            rgba = im.cmap(im.norm(val))
            txt_color = _light_or_dark_text(to_rgb(rgba[:3]))
            # Dataset はインデックス文字列（"OH new vs signless" 等）から先頭2文字で判別
            ds_tag = "OH" if str(ds).strip().startswith("OH") else "FP"
            k_tuple = meta_idx.get((ds_tag, key), (np.nan, np.nan))
            if np.isnan(k_tuple[0]) or np.isnan(k_tuple[1]):
                label = f"{val:.2f}\n(-/-)"
            else:
                label = f"{val:.2f}\n({int(k_tuple[0])}/{int(k_tuple[1])})"
            ax.text(j, i, label, ha="center", va="center", fontsize=8, color=txt_color, linespacing=1.0)

    ax.set_xlabel("Mode–Index")
    ax.set_ylabel("Dataset / Comparison")
    ax.set_title(f"{title}\n{subtitle}", pad=10)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("ARI", rotation=270, labelpad=12)
    fig.tight_layout()
    fig.savefig(out_path_png, bbox_inches="tight"); fig.savefig(out_path_pdf, bbox_inches="tight")
    plt.close(fig)

# ========= メイン =========
def main():
    set_font()

    # new (Main) と signless をOH/FPそれぞれ収集
    results_all = []
    tables_meta = []
    k_summaries = []

    for ds in ["OH","FP"]:
        A = collect_new_labels(FIGS_DIR[ds], ds)          # new: id × [top3_gap, ...]
        S = load_signless_labels(SIGNLESS_DIR[ds])        # signless: id × [top3_gap, ...]
        if A is None or S is None:
            print(f"[WARN] Skip {ds}: A or S not found.")
            continue

        # kの参考表（new側のみ）
        k_summaries.append(summarize_k_from_assign(FIGS_DIR[ds], ds))

        # 一致度
        T = pairwise_scores(A, S)  # condition, n, kA, kB, ARI, NMI, AMI
        if T.empty:
            print(f"[WARN] {ds}: no common conditions."); 
            continue

        # 付加情報
        T["mode"]    = T["condition"].str.replace(r"_(.*)$", "", regex=True)
        T["index"]   = T["condition"].str.replace(r"^(.*)_", "", regex=True)
        T["Pair"]    = "new vs signless"
        T["Dataset"] = ds
        T["k_max"]   = T[["kA","kB"]].max(axis=1)

        results_all.append(T)
        tables_meta.append(T[["Dataset","Pair","condition","kA","kB","n"]])

    if not results_all:
        raise RuntimeError("No results to summarize. Check inputs.")

    T_all = pd.concat(results_all, ignore_index=True)
    META  = pd.concat(tables_meta, ignore_index=True)
    KTAB  = pd.concat(k_summaries, ignore_index=True) if k_summaries else pd.DataFrame(columns=["Dataset","mode","index","k"])

    # 参考：k一覧（new側）
    KTAB.sort_values(["Dataset","mode","index"]).to_csv(OUT_ANALYSIS / "Table_4.2.4.3_k-by-index.csv", index=False, encoding="utf-8-sig")

    # ---------- 本文：k<=30 ----------
    T30 = T_all[(T_all["k_max"].isna()) | (T_all["k_max"] <= K_MAX_KEEP)].copy()
    if len(T30):
        out_csv_main = OUT_MAIN / "Table_4.2.4.3_new-vs-signless_k-le30.csv"
        T30.sort_values(["Dataset","mode","index"]).to_csv(out_csv_main, index=False, encoding="utf-8-sig")

        # heatmap pivot
        pivot_main = (T30.assign(key=lambda d: d["mode"]+"–"+d["index"])
                           .pivot_table(index=["Dataset","Pair"], columns="key", values="ARI", aggfunc="mean", observed=False)
                           .sort_index(axis=0))
        col_order = [f"top3–{ix}" for ix in INDICES] + [f"cumeig–{ix}" for ix in INDICES]
        for c in col_order:
            if c not in pivot_main.columns: pivot_main[c] = np.nan
        pivot_main = pivot_main[col_order]
        png = OUT_MAIN / "Fig_4.2.4.3_new-vs-signless_ARI-heatmap.png"
        pdf = OUT_MAIN / "Fig_4.2.4.3_new-vs-signless_ARI-heatmap.pdf"
        save_heatmap_ari(
            df_pivot=pivot_main.rename(index={("OH","new vs signless"):"OH new vs signless",
                                              ("FP","new vs signless"):"FP new vs signless"}),
            meta_table=META,
            out_path_png=png, out_path_pdf=pdf,
            title="Fig. 4.2.4.3: Clustering agreement (ARI) — new vs signless (k≤30)",
            subtitle="Cell shows ARI with cluster counts (kA/kB). Text color adapts to background."
        )
        print(f"[MAIN] Wrote: {out_csv_main.name}, {png.name}, {pdf.name}")
    else:
        print("[MAIN] No rows for k≤30.")

    # ---------- 補遺：all k ----------
    out_csv_apx = OUT_APPENDIX / "Appendix_Table_allK_new-vs-signless.csv"
    T_all.sort_values(["Dataset","mode","index"]).to_csv(out_csv_apx, index=False, encoding="utf-8-sig")

    pivot_all = (T_all.assign(key=lambda d: d["mode"]+"–"+d["index"])
                      .pivot_table(index=["Dataset","Pair"], columns="key", values="ARI", aggfunc="mean", observed=False)
                      .sort_index(axis=0))
    col_order = [f"top3–{ix}" for ix in INDICES] + [f"cumeig–{ix}" for ix in INDICES]
    for c in col_order:
        if c not in pivot_all.columns: pivot_all[c] = np.nan
    pivot_all = pivot_all[col_order]
    png_apx = OUT_APPENDIX / "Appendix_Fig_allK_new-vs-signless_ARI-heatmap.png"
    pdf_apx = OUT_APPENDIX / "Appendix_Fig_allK_new-vs-signless_ARI-heatmap.pdf"
    save_heatmap_ari(
        df_pivot=pivot_all.rename(index={("OH","new vs signless"):"OH new vs signless",
                                         ("FP","new vs signless"):"FP new vs signless"}),
        meta_table=META,
        out_path_png=png_apx, out_path_pdf=pdf_apx,
        title="Appendix: Clustering agreement (ARI) — new vs signless (all k)",
        subtitle="Cell shows ARI with cluster counts (kA/kB). Text color adapts to background."
    )
    print(f"[APPENDIX] Wrote: {out_csv_apx.name}, {png_apx.name}, {pdf_apx.name}")

    print("\n✅ Done: 4.2.4.3 outputs (main & appendix) →", OUT_ROOT)

if __name__ == "__main__":
    main()

[WARN] OH: no common conditions.
[MAIN] Wrote: Table_4.2.4.3_new-vs-signless_k-le30.csv, Fig_4.2.4.3_new-vs-signless_ARI-heatmap.png, Fig_4.2.4.3_new-vs-signless_ARI-heatmap.pdf
[APPENDIX] Wrote: Appendix_Table_allK_new-vs-signless.csv, Appendix_Fig_allK_new-vs-signless_ARI-heatmap.png, Appendix_Fig_allK_new-vs-signless_ARI-heatmap.pdf

✅ Done: 4.2.4.3 outputs (main & appendix) → /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.3_signless
